# OCT macular: entrenamiento en Google Colab

Pipeline reproducible para un dataset con solo `train/` y `test/`. Test se conserva intacto y se usa únicamente al final.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone https://github.com/magdaleno88/Sam_Oct_IA.git
%cd /content/Sam_Oct_IA
!pip install -q -e .

## Verificar dataset

La estructura esperada es `oct_10/train/<CLASE>/` y `oct_10/test/<CLASE>/`, con clases CNV, DME, DRUSEN y NORMAL.

In [ ]:
from pathlib import Path
DATASET = Path('/content/drive/MyDrive/Sam_Oct_IA/datasets/oct_10')
assert (DATASET / 'train').is_dir(), f'No existe {DATASET / "train"}'
assert (DATASET / 'test').is_dir(), f'No existe {DATASET / "test"}'
for split in ('train', 'test'):
    print(split, {name: len(list((DATASET / split / name).glob('*'))) for name in ('CNV', 'DME', 'DRUSEN', 'NORMAL')})

## Crear manifests

Validation será el 10% de train, estratificado por clase y agrupado por paciente. Si algún nombre no permite inferir paciente, el comando falla; el fallback por imagen solo puede habilitarse explícitamente en `configs/oct.yaml`.

In [ ]:
!python scripts/create_splits.py --config configs/oct.yaml

In [ ]:
import pandas as pd
train = pd.read_csv('data/manifests/train.csv')
val = pd.read_csv('data/manifests/val.csv')
test = pd.read_csv('data/manifests/test.csv')
assert set(train.patient_id.dropna()).isdisjoint(val.patient_id.dropna())
assert set(train.patient_id.dropna()).isdisjoint(test.patient_id.dropna())
assert set(val.patient_id.dropna()).isdisjoint(test.patient_id.dropna())
pd.DataFrame({'split': ['train', 'validation', 'test'], 'images': [len(train), len(val), len(test)]})

## Entrenar ResNet50

El entrenamiento utiliza únicamente train y validation. Test no se carga en `trainer.fit`.

In [ ]:
!python scripts/train_oct.py --config configs/oct.yaml --model improved_resnet50 --experiment colab_oct_10

## Evaluación final sobre test

Este es el único paso que usa test. Guarda métricas, predicciones, matrices de confusión y curvas ROC.

In [ ]:
!python scripts/evaluate.py --run runs/colab_oct_10 --split test --config configs/oct.yaml

In [ ]:
from IPython.display import Image as DisplayImage, display
display(DisplayImage('runs/colab_oct_10/evaluation/test/confusion_matrix.png'))
display(DisplayImage('runs/colab_oct_10/evaluation/test/confusion_matrix_normalized.png'))
display(DisplayImage('runs/colab_oct_10/evaluation/test/roc_curves.png'))